In [14]:
# Building Neural Network With PyTorch

In [15]:
# Introduction to PyTorch and its components

# PyTorch:
    # -An Open-Source deep learning framework that provides flexibility and dynamic computation for building and training Ml models.

# Core Components of PyTorch:
    # 1.Tensors: Multi-Dimensional array similar to Numpy  arrays but with GPU support for accelerated computation.
    # 2.Autograd: Automatic differentiation engine that compute gradients for optimization.
    # 3.torch.nn Module: Provides tools to define and train neural network with layers, activation functions, and loss function.


In [16]:
# Building a Neural Network in PyTorch
# Steps:
    # Define the model:
        # -Use "torch.nn" Module to create a neural network with layers and forward propagation.
    # Define the loss function and optimizer:
        # -Use built-in loss functions like Cross-Entropy Loss
        # -Use optimizers like SGD or Adam for weight updates.

# Training, Evaluating, and Saving a Model in PyTorch

# training:
    # -Forward pass to compute predictions
    # -Compute loss and gradients using backpropagation
    # -Update model weights using the optimizer

# Evaluating:
    # -Assess model performance on a validation or test dataset using metrics like accuracy.
    # -Test the model on unseen data to evaluate its generalization capabilities.
# Saving and loading models: 
    # -Save the trained model using "torch.save" for future use or deployment.
    # -save the model's parameter using torch.save() and load them using torch.load() to restore the model for inference or further training.

In [17]:
# Exercise:
# Objective:
    # Build train, evaluate and save a simple neural network to classify digits from the MNIST digit classification using PyTorch.


In [18]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as f
# Torcn.nn module provides classes and functions for building and defining neural network layers, activation functions, and loss functions. It is a fundamental component of PyTorch for creating and training neural networks.

# define transformations
 # transforms.Compose() is a function that allows you to chain together multiple image transformations. It takes a list of transformation functions as input and applies them sequentially to the input data. For example, you can use transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]) to convert images to tensors and normalize them.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
    # Tranfoms.compose: 
        # 1.combines a list of transformations to be appplied in sequentially to the input data.
    # ToTensor: converts the imgae data into PyTorch tensor which scales the pixel values to the range [0,1].
    # Normalize: normalizes the tensor data by subtracting the mean and dividing by the standard deviation, which helps in improving the convergence of the model during training.
    # or Normalizing tensor data to have a mean of 0.5, and SD of 0.5 for better training stability. 
])

# Load the data set
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

# Create data Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# Here dataloader function loads data in batch for training and testing.
    # batch_size means number of samples per batch, Shuffle  , it shuffle the training data to ensure randomness and prevent overfitting.
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

print(f"training data size: {len(train_dataset)}")
print(f"test data size:{len(test_dataset)}")

# define Model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128) #first input layer with 28*28 input features and 128 output features
        self.fc2 = nn.Linear(128, 64) #second hidden layer with 128 input features and 64 output features
        self.fc3 = nn.Linear(64, 10) #output layer with 64 input features and 10 output features for 10 classes 

    def forward(self, x):
        x = self.flatten(x) #flatten the input image to a 1D vector
        x = f.relu(self.fc1(x)) #apply ReLU activation function to the output of the first layer
        x = f.relu(self.fc2(x)) #apply ReLU activation function to the output of the second layer
        x = self.fc3(x) #output layer without activation function (since we will use CrossEntropyLoss which applies softmax internally)
        return x
# Here we define a simple feedforward neural network with one input layer, two hidden layers, and one output layer. The forward method defines the forward pass of the network, applying ReLU activation functions to the hidden layers and leaving the output layer without an activation function since we will use CrossEntropyLoss which applies softmax internally.
# Instantiate the model, define the loss function and optimizer
model = NeuralNetwork()
print(model)

# define loss functions and optimizer
criterion = nn.CrossEntropyLoss() #loss function for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) #Adam optimizer with learning rate of 0.001
# model.parameters() returns an iterator over the model's parameters, which are the weights and biases of the neural network that will be updated during training. The Adam optimizer is an adaptive learning rate optimization algorithm that adjusts the learning rate for each parameter based on the first and second moments of the gradients, making it efficient for training deep neural networks.

# training loop
def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train() #set the model to training mode
    for epoch in range(epochs):
        running_loss=0.0
        for images, labels in train_loader:
            # Zero gradients
            optimizer.zero_grad()

            # forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"epoch {epoch+1}, loss: {running_loss/len(train_loader): .4f}")

train_model(model, train_loader, criterion, optimizer)

# Evaluate the model
def evaluate_model(model, test_loader):
    model.eval() #set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad(): #disable gradient calculation for evaluation
        for images, label in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1) #get the index of the max log-probability
            total += label.size(0) #increment total count
            correct += (predicted == label).sum().item() #increment correct count
    print(f"test accuracy: {100 * correct/total:.2f}")

evaluate_model(model, test_loader)

# Save the model
torch.save(model.state_dict(), 'mnist_model.pth')

# reload the model
loaded_model = NeuralNetwork()
loaded_model.load_state_dict(torch.load('mnist_model.pth'))

# verify the loaded model
evaluate_model(loaded_model, test_loader)

# update optimizer with new learning rate
# optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
# train_model(model, train_loader, criterian, optimizer)
# evaluate_model(model, test_loader)

training data size: 60000
test data size:10000
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)
epoch 1, loss:  0.3452
epoch 2, loss:  0.1670
epoch 3, loss:  0.1293
epoch 4, loss:  0.1041
epoch 5, loss:  0.0911
test accuracy: 96.41
test accuracy: 96.41
